# Entropy and Volatility Spread with MST-HRP

Uses entropy regimes to switch between MST-HRP, HRP and maximum-Sharpe allocation.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
ENTROPIC VOLATILITY SPREAD STRATEGY - COMPLETE FRAMEWORK
========================================================

STRATEGY CONCEPT:
----------------
Uses Student-t entropy of VIX - Yang-Zhang volatility spread to detect
market dislocations and adjust portfolio exposure accordingly.

ENTROPY INTERPRETATION:
----------------------
- HIGH ENTROPY: Market dislocation → Defensive (40% exposure, MST-HRP)
- MEDIUM ENTROPY: Normal markets → Balanced (80% exposure, HRP)
- LOW ENTROPY: Market alignment → Aggressive (120% exposure, Max Sharpe)

PORTFOLIO OPTIMIZATION BY REGIME:
---------------------------------
HIGH ENTROPY (Defensive):
  - MST clustering of correlation matrix
  - HRP within each cluster
  - Maximum diversification

MEDIUM ENTROPY (Balanced):
  - Standard HRP across all assets
  - Equal risk contribution

LOW ENTROPY (Aggressive):
  - Maximum Sharpe ratio
  - Return maximization with leverage

TRADING FLOW (NO LOOK-AHEAD):
-----------------------------
Day T-1 (Close):
├─ Calculate VIX[T-1] and Yang-Zhang[T-1] using data through T-2
├─ Compute spread z-score from historical distribution
├─ Calculate Student-t entropy
├─ Detect entropy regime
├─ Generate portfolio allocation signal
└─ Store for execution

Day T (Open):
├─ Execute trades at Open[T]
└─ Apply transaction costs

Day T (Close):
├─ Value portfolio at Close[T]
└─ Returns = Close[T]/Close[T-1] - 1

Author: Production Trading System
Date: 2025
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple, Optional, List
from scipy import stats
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.covariance import LedoitWolf
from dataclasses import dataclass

warnings.filterwarnings("ignore")


class EntropyStrategyConfig:
    """Centralized configuration - ALL PARAMETERS CONFIGURABLE"""

    BACKTEST_END_DATE = "2018-12-31"
    BACKTEST_YEARS = 5
    START_DATE = None
    END_DATE = None
    INITIAL_CAPITAL = 100000.0
    BENCHMARK = "URTH"
    YANG_ZHANG_WINDOW = 20
    ZSCORE_LOOKBACK = 252
    ENTROPY_BINS = 50
    STUDENT_T_DF = 5
    MIN_TRAINING_DAYS = 252
    REFIT_FREQUENCY = 63
    HIGH_ENTROPY_THRESHOLD_INIT = 2.5
    LOW_ENTROPY_THRESHOLD_INIT = 1.5
    HIGH_ENTROPY_EXPOSURE = 0.4
    MEDIUM_ENTROPY_EXPOSURE = 0.8
    LOW_ENTROPY_EXPOSURE = 1.2
    HIGH_ENTROPY_ASSETS = ["TLT", "IEF", "GLD", "XLU", "XLP", "USMV", "SH"]
    MEDIUM_ENTROPY_ASSETS = ["SPY", "QQQ", "IWM", "VTI", "EFA", "VNQ", "AGG", "XLV", "XLF", "XLI"]
    LOW_ENTROPY_ASSETS = [
        "QQQ",
        "XLK",
        "VGT",
        "NVDA",
        "MSFT",
        "AAPL",
        "META",
        "GOOGL",
        "AMZN",
        "IWM",
        "MTUM",
    ]
    MAX_POSITION_SIZE = 0.2
    MIN_POSITION_SIZE = 0.05
    OPTIMIZATION_LOOKBACK = 252
    REBALANCE_FREQUENCY = 21
    MIN_REBALANCE_THRESHOLD = 0.05
    COMMISSION_RATE = 0.0005
    BID_ASK_SPREAD = 0.0003
    MARKET_IMPACT = 0.0002
    MIN_COMMISSION = 1.0
    MAX_COMMISSION = 50.0
    STOP_LOSS_ENABLED = True
    STOP_LOSS_THRESHOLD = -0.15
    MAX_DRAWDOWN_PAUSE = -0.2
    RISK_FREE_RATE = 0.02

    def __post_init__(self):
        """Calculate derived parameters"""
        end_dt = datetime.strptime(self.BACKTEST_END_DATE, "%Y-%m-%d")
        start_dt = end_dt - timedelta(days=365 * self.BACKTEST_YEARS)
        self.END_DATE = self.BACKTEST_END_DATE
        self.START_DATE = start_dt.strftime("%Y-%m-%d")


class DataFetcher:
    """Data acquisition from Yahoo Finance - NO SYNTHETIC DATA"""

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch OHLCV data from Yahoo Finance.
        NO bfill/ffill - returns only actual data.
        """
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data or not data["historical"]:
                raise ValueError(f"No data for {ticker}")
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df = df.rename(
                columns={
                    "date": "Date",
                    "open": "Open",
                    "high": "High",
                    "low": "Low",
                    "close": "Close",
                    "volume": "Volume",
                }
            )
            df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]
            if len(df) < 100:
                raise ValueError(f"Insufficient data for {ticker}")
            print(f"  ✓ {ticker}: {len(df)} days")
            return df
        except Exception as e:
            print(f"  ✗ Error {ticker}: {str(e)}")
            raise


class EntropyCalculator:
    """
    Calculate Student-t entropy of VIX - Yang-Zhang spread.

    Uses Student-t for fat-tail modeling of financial volatility.
    """

    def __init__(self, student_t_df: int = 5, num_bins: int = 50):
        self.df = student_t_df
        self.num_bins = num_bins

    def calculate_yang_zhang_volatility(self, ohlc_df: pd.DataFrame, window: int) -> pd.Series:
        """
        Yang-Zhang volatility using OHLC data.

        ANTI-LOOK-AHEAD: Uses data through T-2 for volatility at T-1.

        Formula: σ² = σ_open² + k*σ_close² + (1-k)*σ_rs²
        """
        O = ohlc_df["Open"]
        H = ohlc_df["High"]
        L = ohlc_df["Low"]
        C = ohlc_df["Close"]
        valid_mask = (L <= O) & (O <= H) & (L <= C) & (C <= H)
        log_ho = np.log(H / O.replace(0, np.nan))
        log_lo = np.log(L / O.replace(0, np.nan))
        log_co = np.log(C / O.replace(0, np.nan))
        log_oc = np.log(O / C.shift(1).replace(0, np.nan))
        rs_vol = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)
        close_vol = log_co**2
        open_vol = log_oc**2
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        rs_roll = rs_vol.rolling(window, min_periods=window // 2).mean()
        close_roll = close_vol.rolling(window, min_periods=window // 2).mean()
        open_roll = open_vol.rolling(window, min_periods=window // 2).mean()
        yz_vol = np.sqrt(open_roll + k * close_roll + (1 - k) * rs_roll) * np.sqrt(252)
        yz_vol = yz_vol.where(valid_mask, np.nan)
        return yz_vol

    def calculate_entropy(self, z_scores: np.ndarray) -> float:
        """
        Calculate Shannon entropy from z-score distribution.

        H = -Σ p(z) log(p(z))

        Higher entropy = more uncertainty = flatter distribution
        """
        z_clean = z_scores[np.isfinite(z_scores)]
        if len(z_clean) < 10:
            return np.nan
        try:
            counts, bin_edges = np.histogram(z_clean, bins=self.num_bins)
            probabilities = counts / counts.sum()
            probabilities = probabilities[probabilities > 0]
            entropy = -np.sum(probabilities * np.log(probabilities))
            return entropy
        except:
            return np.nan


@dataclass
class EntropyThresholds:
    """Stores entropy thresholds from training period"""

    high_entropy_threshold: float
    low_entropy_threshold: float
    training_end_date: pd.Timestamp
    training_size: int
    mean_entropy: float
    std_entropy: float


class WalkForwardEntropyDetector:
    """
    Walk-forward entropy regime detector with NO LOOK-AHEAD.

    Guarantees:
    - VIX lagged by 1 day
    - Yang-Zhang uses data through T-2
    - Z-score from historical distribution
    - Thresholds from training period only
    """

    def __init__(self, config: EntropyStrategyConfig):
        self.config = config
        self.entropy_calc = EntropyCalculator(config.STUDENT_T_DF, config.ENTROPY_BINS)
        self.regime_names = {
            2: "High Entropy (Defensive)",
            1: "Medium Entropy (Balanced)",
            0: "Low Entropy (Aggressive)",
        }

    def calculate_features_t_minus_1(
        self, spy_ohlc: pd.DataFrame, vix_close: pd.Series
    ) -> pd.DataFrame:
        """
        Calculate features with STRICT T-1 methodology.

        Process:
        1. Yang-Zhang from OHLC (uses T-2)
        2. Align VIX
        3. Calculate spread
        4. Lag all features by 1 day
        """
        print("\n" + "=" * 80)
        print("CALCULATING FEATURES (T-1 METHODOLOGY)")
        print("=" * 80)
        df = spy_ohlc.copy().sort_values("Date").reset_index(drop=True)
        print(f"\n1. Yang-Zhang volatility (window={self.config.YANG_ZHANG_WINDOW})...")
        df["YZ_vol"] = self.entropy_calc.calculate_yang_zhang_volatility(
            df, self.config.YANG_ZHANG_WINDOW
        )
        print("\n2. Aligning VIX data...")
        vix_aligned = [
            vix_close.loc[date] if date in vix_close.index else np.nan for date in df["Date"]
        ]
        df["VIX"] = vix_aligned
        print("\n3. Calculating spread...")
        df["Spread"] = df["VIX"] / 100.0 - df["YZ_vol"]
        print("\n4. Applying lag...")
        df["VIX_lag"] = df["VIX"].shift(1)
        df["YZ_vol_lag"] = df["YZ_vol"].shift(1)
        df["Spread_lag"] = df["Spread"].shift(1)
        print(f"\n✓ Features ready: {df['Spread_lag'].notna().sum()} observations")
        return df

    def calculate_zscore_and_entropy(
        self, features_df: pd.DataFrame, current_idx: int, lookback: int
    ) -> Tuple[float, float]:
        """
        Calculate z-score and entropy using ONLY historical data.

        Uses expanding window up to current_idx (exclusive).
        """
        hist_start = max(0, current_idx - lookback)
        hist_spreads = features_df.iloc[hist_start:current_idx]["Spread_lag"].dropna()
        if len(hist_spreads) < 20:
            return (np.nan, np.nan)
        hist_mean = hist_spreads.mean()
        hist_std = hist_spreads.std()
        if hist_std == 0 or pd.isna(hist_std):
            return (np.nan, np.nan)
        current_spread = features_df.loc[current_idx, "Spread_lag"]
        if pd.isna(current_spread):
            return (np.nan, np.nan)
        z_score = (current_spread - hist_mean) / hist_std
        historical_z_scores = (hist_spreads - hist_mean) / hist_std
        entropy = self.entropy_calc.calculate_entropy(historical_z_scores.values)
        return (z_score, entropy)

    def calculate_entropy_thresholds(
        self, features_df: pd.DataFrame, end_idx: int
    ) -> EntropyThresholds:
        """
        Calculate thresholds using EXPANDING WINDOW.

        Uses 75th/25th percentiles of historical entropy.
        """
        entropy_values = []
        for i in range(self.config.ZSCORE_LOOKBACK, end_idx):
            _, entropy = self.calculate_zscore_and_entropy(
                features_df, i, self.config.ZSCORE_LOOKBACK
            )
            if not pd.isna(entropy):
                entropy_values.append(entropy)
        if len(entropy_values) < 50:
            print(f"   ! Insufficient entropy data, using defaults")
            return EntropyThresholds(
                high_entropy_threshold=self.config.HIGH_ENTROPY_THRESHOLD_INIT,
                low_entropy_threshold=self.config.LOW_ENTROPY_THRESHOLD_INIT,
                training_end_date=features_df.iloc[end_idx - 1]["Date"],
                training_size=end_idx,
                mean_entropy=np.nan,
                std_entropy=np.nan,
            )
        entropy_array = np.array(entropy_values)
        high_threshold = np.percentile(entropy_array, 75)
        low_threshold = np.percentile(entropy_array, 25)
        return EntropyThresholds(
            high_entropy_threshold=high_threshold,
            low_entropy_threshold=low_threshold,
            training_end_date=features_df.iloc[end_idx - 1]["Date"],
            training_size=end_idx,
            mean_entropy=entropy_array.mean(),
            std_entropy=entropy_array.std(),
        )

    def classify_entropy_regime(
        self, entropy: float, thresholds: EntropyThresholds
    ) -> Tuple[int, str]:
        """
        Classify entropy into regime.

        HIGH (>75th): Defensive
        MEDIUM (25th-75th): Balanced
        LOW (<25th): Aggressive
        """
        if pd.isna(entropy):
            return (1, self.regime_names[1])
        if entropy > thresholds.high_entropy_threshold:
            regime_idx = 2
        elif entropy < thresholds.low_entropy_threshold:
            regime_idx = 0
        else:
            regime_idx = 1
        return (regime_idx, self.regime_names[regime_idx])

    def walk_forward_detect(self, features_df: pd.DataFrame) -> Tuple[pd.DataFrame, List[Dict]]:
        """
        Execute walk-forward entropy detection.

        Process:
        - Train thresholds on expanding window
        - Predict regimes using frozen thresholds
        - Refit every REFIT_FREQUENCY days
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD ENTROPY DETECTION")
        print("=" * 80)
        df_clean = features_df.dropna(subset=["Spread_lag"]).copy().reset_index(drop=True)
        print(f"\nObservations: {len(df_clean)}")
        print(f"Range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")
        df_clean["ZScore"] = np.nan
        df_clean["Entropy"] = np.nan
        df_clean["Regime_Idx"] = np.nan
        df_clean["Regime_Name"] = None
        df_clean["Entropy_Threshold_High"] = np.nan
        df_clean["Entropy_Threshold_Low"] = np.nan
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        print(f"\nConfig:")
        print(f"  Min training: {min_train} days")
        print(f"  Refit: {refit_freq} days")
        fold_list = []
        fold_num = 0
        last_refit_idx = min_train
        current_thresholds = None
        for i in range(min_train, len(df_clean)):
            days_since_refit = i - last_refit_idx
            if current_thresholds is None or days_since_refit >= refit_freq:
                fold_num += 1
                train_end = i
                print(f"\n{'=' * 80}")
                print(f"FOLD {fold_num}: Refitting at index {i}")
                print(f"{'=' * 80}")
                print(f"Training: [0:{train_end}] EXPANDING")
                print(
                    f"Dates: {df_clean['Date'].iloc[0].date()} to {df_clean['Date'].iloc[train_end - 1].date()}"
                )
                current_thresholds = self.calculate_entropy_thresholds(df_clean, train_end)
                print(f"\nThresholds:")
                print(f"  HIGH: {current_thresholds.high_entropy_threshold:.4f}")
                print(f"  LOW: {current_thresholds.low_entropy_threshold:.4f}")
                fold_list.append(
                    {
                        "fold_num": fold_num,
                        "train_end": train_end,
                        "train_end_date": df_clean["Date"].iloc[train_end - 1],
                        "high_threshold": current_thresholds.high_entropy_threshold,
                        "low_threshold": current_thresholds.low_entropy_threshold,
                    }
                )
                last_refit_idx = i
            z_score, entropy = self.calculate_zscore_and_entropy(
                df_clean, i, self.config.ZSCORE_LOOKBACK
            )
            regime_idx, regime_name = self.classify_entropy_regime(entropy, current_thresholds)
            df_clean.loc[i, "ZScore"] = z_score
            df_clean.loc[i, "Entropy"] = entropy
            df_clean.loc[i, "Regime_Idx"] = regime_idx
            df_clean.loc[i, "Regime_Name"] = regime_name
            df_clean.loc[i, "Entropy_Threshold_High"] = current_thresholds.high_entropy_threshold
            df_clean.loc[i, "Entropy_Threshold_Low"] = current_thresholds.low_entropy_threshold
        print(f"\n{'=' * 80}")
        print("DETECTION COMPLETE")
        print(f"{'=' * 80}")
        print(f"Folds: {len(fold_list)}")
        print(f"Predictions: {df_clean['Regime_Name'].notna().sum()}")
        regime_counts = df_clean["Regime_Name"].value_counts()
        print(f"\nRegime distribution:")
        for regime in self.regime_names.values():
            count = regime_counts.get(regime, 0)
            pct = count / len(df_clean) * 100 if len(df_clean) > 0 else 0
            print(f"  {regime:30s}: {count:4d} ({pct:5.1f}%)")
        return (df_clean, fold_list)


class RegimeBasedOptimizer:
    """
    Regime-specific optimization:

    HIGH ENTROPY → MST clustering + HRP per cluster (defensive)
    MEDIUM ENTROPY → Standard HRP (balanced)
    LOW ENTROPY → Max Sharpe (aggressive)
    """

    def __init__(self, config: EntropyStrategyConfig):
        self.config = config
        self.risk_free_rate = config.RISK_FREE_RATE
        self.min_cluster_size = 2

    def build_mst_clusters(
        self, correlation_matrix: pd.DataFrame, max_clusters: int = 4
    ) -> List[List[str]]:
        """
        Build MST and extract clusters via hierarchical clustering.

        Distance = sqrt(0.5 * (1 - correlation))
        """
        n_assets = len(correlation_matrix)
        if n_assets < 4:
            return [list(correlation_matrix.index)]
        distance_matrix = np.sqrt(0.5 * (1 - correlation_matrix.values))
        condensed_dist = squareform(distance_matrix)
        linkage_matrix = linkage(condensed_dist, method="ward")
        n_clusters = min(max_clusters, max(2, n_assets // 3))
        cluster_labels = fcluster(linkage_matrix, n_clusters, criterion="maxclust")
        clusters = []
        for cluster_id in range(1, n_clusters + 1):
            cluster_assets = [
                correlation_matrix.index[i]
                for i, label in enumerate(cluster_labels)
                if label == cluster_id
            ]
            if len(cluster_assets) >= self.min_cluster_size:
                clusters.append(cluster_assets)
        return clusters if clusters else [list(correlation_matrix.index)]

    def hrp_single_cluster(
        self, returns: pd.DataFrame, target_allocation: float
    ) -> Dict[str, float]:
        """
        Apply HRP to a single cluster.

        Process:
        1. Correlation distance
        2. Hierarchical clustering
        3. Quasi-diagonalization
        4. Recursive bisection
        """
        if len(returns.columns) == 0:
            return {}
        if len(returns.columns) == 1:
            return {returns.columns[0]: target_allocation}
        try:
            corr_matrix = returns.corr()
            distance_matrix = np.sqrt(0.5 * (1 - corr_matrix))
            condensed_distances = squareform(distance_matrix.values)
            linkage_matrix = linkage(condensed_distances, method="ward")
            sorted_indices = self._get_quasi_diagonal(linkage_matrix, len(corr_matrix))
            sorted_corr = corr_matrix.iloc[sorted_indices, sorted_indices]
            hrp_weights = self._get_recursive_bisection(sorted_corr)
            weight_dict = {}
            for i, asset in enumerate(corr_matrix.index):
                original_index = sorted_indices.index(i)
                weight = hrp_weights[original_index] * target_allocation
                if weight > self.config.MIN_POSITION_SIZE:
                    weight_dict[asset] = weight
            return weight_dict
        except Exception as e:
            print(f"  ! HRP error: {e}")
            equal_weight = target_allocation / len(returns.columns)
            return {
                asset: equal_weight
                for asset in returns.columns
                if equal_weight > self.config.MIN_POSITION_SIZE
            }

    def optimize_mst_hrp_clusters(
        self, returns: pd.DataFrame, target_exposure: float
    ) -> Dict[str, float]:
        """
        MST clustering + HRP per cluster (DEFENSIVE).

        Algorithm:
        1. Build MST clusters
        2. Equal allocation across clusters
        3. HRP within each cluster
        """
        if len(returns) < 60 or len(returns.columns) < 3:
            return {}
        print(f"\n  MST-HRP CLUSTERING")
        print(f"  Assets: {len(returns.columns)}")
        try:
            correlation_matrix = returns.corr()
            clusters = self.build_mst_clusters(correlation_matrix, max_clusters=4)
            print(f"  Clusters: {len(clusters)}")
            for i, cluster in enumerate(clusters):
                print(f"    C{i + 1}: {len(cluster)} assets")
            cluster_allocation = target_exposure / len(clusters)
            final_weights = {}
            for cluster_assets in clusters:
                cluster_returns = returns[cluster_assets]
                cluster_weights = self.hrp_single_cluster(cluster_returns, cluster_allocation)
                final_weights.update(cluster_weights)
            total_weight = sum(final_weights.values())
            if total_weight > 0:
                scale = target_exposure / total_weight
                final_weights = {k: v * scale for k, v in final_weights.items()}
            print(f"  Portfolio: {len(final_weights)} assets")
            print(f"  Exposure: {sum(final_weights.values()):.1%}")
            return final_weights
        except Exception as e:
            print(f"  ! MST-HRP error: {e}")
            return {}

    def optimize_hrp_standard(
        self, returns: pd.DataFrame, target_exposure: float
    ) -> Dict[str, float]:
        """
        Standard HRP (BALANCED).

        Single-level hierarchical risk parity across all assets.
        """
        if len(returns) < 60:
            return {}
        print(f"\n  STANDARD HRP")
        print(f"  Assets: {len(returns.columns)}")
        try:
            corr_matrix = returns.corr()
            distance_matrix = np.sqrt(0.5 * (1 - corr_matrix))
            condensed_distances = squareform(distance_matrix.values)
            linkage_matrix = linkage(condensed_distances, method="ward")
            sorted_indices = self._get_quasi_diagonal(linkage_matrix, len(corr_matrix))
            sorted_corr = corr_matrix.iloc[sorted_indices, sorted_indices]
            hrp_weights = self._get_recursive_bisection(sorted_corr)
            weight_dict = {}
            for i, asset in enumerate(corr_matrix.index):
                original_index = sorted_indices.index(i)
                weight = hrp_weights[original_index] * target_exposure
                if weight > self.config.MIN_POSITION_SIZE:
                    weight_dict[asset] = weight
            print(f"  Portfolio: {len(weight_dict)} assets")
            print(f"  Exposure: {sum(weight_dict.values()):.1%}")
            return weight_dict
        except Exception as e:
            print(f"  ! HRP error: {e}")
            return {}

    def optimize_max_sharpe(
        self, returns: pd.DataFrame, target_exposure: float, max_weight: float
    ) -> Dict[str, float]:
        """
        Maximum Sharpe ratio (AGGRESSIVE).

        Objective: max (Return - RiskFree) / Volatility
        """
        if len(returns) < 60:
            return {}
        print(f"\n  MAX SHARPE")
        print(f"  Assets: {len(returns.columns)}")
        print(f"  Target: {target_exposure:.1%}")
        expected_returns = returns.mean() * 252
        cov_matrix = returns.cov() * 252
        try:
            lw = LedoitWolf()
            cov_shrunk = lw.fit(returns.values).covariance_ * 252
            cov_matrix = pd.DataFrame(cov_shrunk, index=returns.columns, columns=returns.columns)
        except:
            pass
        n_assets = len(expected_returns)

        def neg_sharpe(weights):
            port_return = np.dot(weights, expected_returns)
            port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            return (
                -(port_return - self.risk_free_rate) / port_vol if port_vol > 0 else 10000000000.0
            )

        constraints = [{"type": "eq", "fun": lambda x: np.sum(x) - target_exposure}]
        bounds = [(0, max_weight) for _ in range(n_assets)]
        x0 = np.array([target_exposure / n_assets] * n_assets)
        try:
            result = minimize(
                neg_sharpe,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-09},
            )
            if result.success:
                weights = result.x
                weights = np.where(weights < self.config.MIN_POSITION_SIZE, 0, weights)
                if np.sum(weights) > 0:
                    weights = weights * target_exposure / np.sum(weights)
                    weight_dict = {
                        asset: w
                        for asset, w in zip(expected_returns.index, weights)
                        if w > self.config.MIN_POSITION_SIZE
                    }
                    print(f"  Portfolio: {len(weight_dict)} assets")
                    print(f"  Exposure: {sum(weight_dict.values()):.1%}")
                    return weight_dict
        except Exception as e:
            print(f"  ! Optimization error: {e}")
        return {}

    def optimize_by_regime(self, regime_idx: int, returns: pd.DataFrame) -> Dict[str, float]:
        """
        Route to regime-specific optimizer.

        HIGH (2) → MST-HRP clusters
        MEDIUM (1) → Standard HRP
        LOW (0) → Max Sharpe
        """
        if len(returns) < 60:
            return {}
        try:
            if regime_idx == 2:
                return self.optimize_mst_hrp_clusters(returns, self.config.HIGH_ENTROPY_EXPOSURE)
            elif regime_idx == 1:
                return self.optimize_hrp_standard(returns, self.config.MEDIUM_ENTROPY_EXPOSURE)
            else:
                return self.optimize_max_sharpe(
                    returns, self.config.LOW_ENTROPY_EXPOSURE, self.config.MAX_POSITION_SIZE
                )
        except Exception as e:
            print(f"  ! Error regime {regime_idx}: {e}")
            return {}

    def _get_quasi_diagonal(self, linkage_matrix, num_assets):
        """Get quasi-diagonal order from clustering"""
        sorted_indices = []

        def append_order(node_index):
            if node_index < num_assets:
                sorted_indices.append(node_index)
            else:
                left = int(linkage_matrix[node_index - num_assets][0])
                right = int(linkage_matrix[node_index - num_assets][1])
                append_order(left)
                append_order(right)

        append_order(len(linkage_matrix) + num_assets - 1)
        return sorted_indices

    def _get_recursive_bisection(self, corr_matrix):
        """Recursive bisection for HRP weights"""
        weights = pd.Series(1.0, index=corr_matrix.index)
        clustered_alphas = [corr_matrix.index.tolist()]
        while len(clustered_alphas) > 0:
            clustered_alphas = [
                cluster[start:end]
                for cluster in clustered_alphas
                for start, end in ((0, len(cluster) // 2), (len(cluster) // 2, len(cluster)))
                if len(cluster) > 1
            ]
            for i in range(0, len(clustered_alphas), 2):
                if i + 1 < len(clustered_alphas):
                    left = clustered_alphas[i]
                    right = clustered_alphas[i + 1]
                    left_var = self._get_cluster_var(corr_matrix, left)
                    right_var = self._get_cluster_var(corr_matrix, right)
                    total_var = left_var + right_var
                    if total_var > 0:
                        left_weight = right_var / total_var
                        right_weight = left_var / total_var
                        for asset in left:
                            weights[asset] *= left_weight
                        for asset in right:
                            weights[asset] *= right_weight
        return weights.values

    def _get_cluster_var(self, corr_matrix, cluster):
        """Calculate cluster variance"""
        if len(cluster) == 1:
            return 1.0
        cluster_corr = corr_matrix.loc[cluster, cluster]
        weights = np.ones(len(cluster)) / len(cluster)
        return np.dot(weights.T, np.dot(cluster_corr.values, weights))


class BacktestEngine:
    """
    Realistic backtesting with proper execution flow.

    T-1 Close → Signal
    T Open → Execute
    T Close → Valuation
    """

    def __init__(self, config: EntropyStrategyConfig):
        self.config = config
        self.optimizer = RegimeBasedOptimizer(config)
        self.trades = []

    def calculate_transaction_costs(self, trade_value: float) -> float:
        """Calculate realistic costs: commission + spread + impact"""
        abs_value = abs(trade_value)
        commission = np.clip(
            abs_value * self.config.COMMISSION_RATE,
            self.config.MIN_COMMISSION,
            self.config.MAX_COMMISSION,
        )
        spread = abs_value * self.config.BID_ASK_SPREAD
        impact = abs_value * self.config.MARKET_IMPACT
        return commission + spread + impact

    def run_backtest(
        self,
        regime_predictions: pd.DataFrame,
        asset_data: Dict[str, pd.DataFrame],
        initial_capital: float,
    ) -> Dict:
        """Execute backtest with regime-based allocation"""
        print("\n" + "=" * 80)
        print("EXECUTING BACKTEST")
        print("=" * 80)
        valid_regimes = regime_predictions.dropna(subset=["Regime_Idx"])
        trading_dates = valid_regimes["Date"].iloc[:-1]
        print(f"\nPeriod: {trading_dates.iloc[0].date()} to {trading_dates.iloc[-1].date()}")
        print(f"Days: {len(trading_dates)}")
        portfolio_value = initial_capital
        cash = initial_capital
        positions = {}
        portfolio_history = []
        total_costs = 0.0
        regime_universes = {
            2: self.config.HIGH_ENTROPY_ASSETS,
            1: self.config.MEDIUM_ENTROPY_ASSETS,
            0: self.config.LOW_ENTROPY_ASSETS,
        }
        last_rebalance = -self.config.REBALANCE_FREQUENCY
        for day_idx, signal_date in enumerate(trading_dates):
            if day_idx % 50 == 0:
                print(
                    f"  Day {day_idx + 1}/{len(trading_dates)}: {signal_date.strftime('%Y-%m-%d')}"
                )
            regime_row = regime_predictions[regime_predictions["Date"] == signal_date]
            if regime_row.empty:
                continue
            regime_idx = int(regime_row["Regime_Idx"].iloc[0])
            regime_name = regime_row["Regime_Name"].iloc[0]
            entropy = regime_row["Entropy"].iloc[0]
            current_prices = {}
            for ticker in positions.keys():
                if ticker in asset_data:
                    df = asset_data[ticker]
                    matches = df[df["Date"].dt.date == signal_date.date()]
                    if not matches.empty:
                        current_prices[ticker] = matches.iloc[0]["Close"]
            risky_value = sum(
                (
                    positions.get(ticker, 0) * current_prices.get(ticker, 0)
                    for ticker in positions.keys()
                )
            )
            if day_idx > 0:
                cash += cash * (self.config.RISK_FREE_RATE / 252)
            portfolio_value = cash + risky_value
            days_since_rebal = day_idx - last_rebalance
            should_rebalance = days_since_rebal >= self.config.REBALANCE_FREQUENCY
            if should_rebalance and day_idx >= self.config.OPTIMIZATION_LOOKBACK:
                universe = regime_universes[regime_idx]
                available_assets = [a for a in universe if a in asset_data]
                if len(available_assets) < 3:
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                hist_start = max(0, day_idx - self.config.OPTIMIZATION_LOOKBACK)
                hist_dates = trading_dates.iloc[hist_start:day_idx]
                returns_dict = {}
                for asset in available_assets:
                    if asset not in asset_data:
                        continue
                    df = asset_data[asset]
                    asset_returns = []
                    for date in hist_dates:
                        matches = df[df["Date"].dt.date == date.date()]
                        if not matches.empty:
                            asset_returns.append(matches.iloc[0]["Close"])
                        else:
                            asset_returns.append(np.nan)
                    if len(asset_returns) > 0:
                        returns_series = pd.Series(asset_returns).pct_change()
                        if returns_series.notna().sum() / len(returns_series) >= 0.8:
                            returns_dict[asset] = returns_series.values
                if len(returns_dict) < 3:
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                returns_matrix = pd.DataFrame(returns_dict, index=hist_dates)
                returns_matrix = returns_matrix.dropna(how="all")
                target_weights = self.optimizer.optimize_by_regime(regime_idx, returns_matrix)
                if not target_weights:
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                next_idx = day_idx + 1
                if next_idx >= len(trading_dates):
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                exec_date = trading_dates.iloc[next_idx]
                exec_prices = {}
                for ticker in target_weights.keys():
                    if ticker in asset_data:
                        df = asset_data[ticker]
                        matches = df[df["Date"].dt.date == exec_date.date()]
                        if not matches.empty:
                            exec_prices[ticker] = matches.iloc[0]["Open"]
                if not exec_prices:
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                trade_costs = 0.0
                for ticker, weight in target_weights.items():
                    if ticker in exec_prices:
                        target_value = portfolio_value * weight
                        current_value = positions.get(ticker, 0) * exec_prices[ticker]
                        trade_value = target_value - current_value
                        if abs(trade_value) > portfolio_value * 0.01:
                            cost = self.calculate_transaction_costs(trade_value)
                            shares = trade_value / exec_prices[ticker]
                            positions[ticker] = positions.get(ticker, 0) + shares
                            cash -= trade_value + cost
                            trade_costs += cost
                            self.trades.append(
                                {
                                    "Date": exec_date,
                                    "Ticker": ticker,
                                    "Action": "BUY" if trade_value > 0 else "SELL",
                                    "Value": abs(trade_value),
                                    "Price": exec_prices[ticker],
                                    "Cost": cost,
                                    "Regime": regime_name,
                                }
                            )
                for ticker in list(positions.keys()):
                    if ticker not in target_weights and ticker in exec_prices:
                        if positions[ticker] != 0:
                            value = positions[ticker] * exec_prices[ticker]
                            cost = self.calculate_transaction_costs(abs(value))
                            cash += value - cost
                            positions[ticker] = 0
                            trade_costs += cost
                            self.trades.append(
                                {
                                    "Date": exec_date,
                                    "Ticker": ticker,
                                    "Action": "CLOSE",
                                    "Value": abs(value),
                                    "Price": exec_prices[ticker],
                                    "Cost": cost,
                                    "Regime": regime_name,
                                }
                            )
                total_costs += trade_costs
                last_rebalance = day_idx
            portfolio_history.append(
                {
                    "Date": signal_date,
                    "Value": portfolio_value,
                    "Cash": cash,
                    "Risky": risky_value,
                    "Regime": regime_name,
                    "Entropy": entropy,
                    "Regime_Idx": regime_idx,
                }
            )
        print("\n" + "=" * 80)
        print("CALCULATING METRICS")
        print("=" * 80)
        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df["Return"] = portfolio_df["Value"].pct_change()
        final_value = portfolio_df["Value"].iloc[-1]
        total_return = final_value / initial_capital - 1
        days = len(portfolio_df)
        annual_return = (1 + total_return) ** (252 / days) - 1
        returns = portfolio_df["Return"].dropna()
        volatility = returns.std() * np.sqrt(252)
        sharpe = (annual_return - self.config.RISK_FREE_RATE) / volatility if volatility > 0 else 0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()
        metrics = {
            "initial_capital": initial_capital,
            "final_value": final_value,
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe,
            "max_drawdown": max_dd,
            "total_costs": total_costs,
            "num_trades": len(self.trades),
        }
        print(f"\n{'=' * 80}")
        print("BACKTEST COMPLETE")
        print(f"{'=' * 80}")
        print(f"Final Value: ${final_value:,.0f}")
        print(f"Total Return: {total_return:.1%}")
        print(f"Annual Return: {annual_return:.1%}")
        print(f"Sharpe: {sharpe:.2f}")
        print(f"Max DD: {max_dd:.1%}")
        print(f"Costs: ${total_costs:,.0f}")
        print(f"Trades: {len(self.trades)}")
        return {"portfolio_history": portfolio_df, "trade_history": self.trades, "metrics": metrics}


def main():
    """Execute Entropic Volatility Spread Strategy"""
    print("=" * 80)
    print("ENTROPIC VOLATILITY SPREAD STRATEGY")
    print("=" * 80)
    print("\nConcept: Entropy of VIX - Yang-Zhang spread")
    print("  High entropy → Defensive (MST-HRP)")
    print("  Medium entropy → Balanced (HRP)")
    print("  Low entropy → Aggressive (Max Sharpe)")
    config = EntropyStrategyConfig()
    config.__post_init__()
    print(f"\nConfig:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Benchmark: {config.BENCHMARK}")
    print("\n" + "=" * 80)
    print("DATA ACQUISITION")
    print("=" * 80)
    fetcher = DataFetcher()
    buffer_start = (
        datetime.strptime(config.START_DATE, "%Y-%m-%d") - timedelta(days=300)
    ).strftime("%Y-%m-%d")
    print(f"\nFetching SPY (buffer from {buffer_start})...")
    spy_data = fetcher.fetch_historical_data("SPY", buffer_start, config.END_DATE)
    print("\nFetching VIX...")
    vix_data = fetcher.fetch_historical_data("^VIX", buffer_start, config.END_DATE)
    print("\nFetching asset universe...")
    all_assets = list(
        set(
            config.HIGH_ENTROPY_ASSETS
            + config.MEDIUM_ENTROPY_ASSETS
            + config.LOW_ENTROPY_ASSETS
            + [config.BENCHMARK]
        )
    )
    asset_data = {}
    for ticker in all_assets:
        try:
            df = fetcher.fetch_historical_data(ticker, buffer_start, config.END_DATE)
            asset_data[ticker] = df
        except:
            print(f"  ! Skipping {ticker}")
    print(f"\n✓ {len(asset_data)} assets fetched")
    print("\n" + "=" * 80)
    print("ENTROPY REGIME DETECTION")
    print("=" * 80)
    detector = WalkForwardEntropyDetector(config)
    vix_series = vix_data.set_index("Date")["Close"]
    features_df = detector.calculate_features_t_minus_1(spy_data, vix_series)
    regime_predictions, fold_list = detector.walk_forward_detect(features_df)
    print("\n" + "=" * 80)
    print("BACKTEST EXECUTION")
    print("=" * 80)
    backtest_start = pd.to_datetime(config.START_DATE)
    backtest_end = pd.to_datetime(config.END_DATE)
    backtest_data = regime_predictions[
        (regime_predictions["Date"] >= backtest_start)
        & (regime_predictions["Date"] <= backtest_end)
    ].copy()
    print(f"\nBacktest data: {len(backtest_data)} days")
    engine = BacktestEngine(config)
    results = engine.run_backtest(backtest_data, asset_data, config.INITIAL_CAPITAL)
    print("\n" + "=" * 80)
    print("GENERATING VISUALIZATIONS")
    print("=" * 80)
    fig, axes = plt.subplots(4, 1, figsize=(16, 14))
    ax1 = axes[0]
    entropy_data = backtest_data.dropna(subset=["Entropy"])
    ax1.plot(
        entropy_data["Date"],
        entropy_data["Entropy"],
        linewidth=1.5,
        color="darkblue",
        label="Entropy",
    )
    if "Entropy_Threshold_High" in entropy_data.columns:
        ax1.plot(
            entropy_data["Date"],
            entropy_data["Entropy_Threshold_High"],
            linestyle="--",
            color="red",
            alpha=0.5,
            label="High Threshold",
        )
        ax1.plot(
            entropy_data["Date"],
            entropy_data["Entropy_Threshold_Low"],
            linestyle="--",
            color="green",
            alpha=0.5,
            label="Low Threshold",
        )
    ax1.set_title("Student-t Entropy of VIX - Yang-Zhang Spread", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Entropy", fontsize=11)
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)
    ax2 = axes[1]
    regime_colors = {0: "green", 1: "orange", 2: "red"}
    for regime_idx in [0, 1, 2]:
        mask = backtest_data["Regime_Idx"] == regime_idx
        if mask.sum() > 0:
            ax2.scatter(
                backtest_data.loc[mask, "Date"],
                backtest_data.loc[mask, "Regime_Idx"],
                c=regime_colors[regime_idx],
                alpha=0.6,
                s=20,
                label=detector.regime_names[regime_idx],
            )
    ax2.set_title("Entropy Regime Classification", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Regime", fontsize=11)
    ax2.set_yticks([0, 1, 2])
    ax2.set_yticklabels(["Low\n(Aggressive)", "Medium\n(Balanced)", "High\n(Defensive)"])
    ax2.legend(loc="upper left", fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax3 = axes[2]
    portfolio_df = results["portfolio_history"]
    ax3.plot(
        portfolio_df["Date"],
        portfolio_df["Value"],
        linewidth=2.5,
        color="darkblue",
        label="Strategy",
    )
    if config.BENCHMARK in asset_data:
        bench_df = asset_data[config.BENCHMARK]
        bench_aligned = []
        for date in portfolio_df["Date"]:
            matches = bench_df[bench_df["Date"].dt.date == date.date()]
            if not matches.empty:
                bench_aligned.append({"Date": date, "Close": matches.iloc[0]["Close"]})
        if bench_aligned:
            bench_plot = pd.DataFrame(bench_aligned)
            first_close = bench_plot["Close"].iloc[0]
            bench_plot["Value"] = bench_plot["Close"] / first_close * config.INITIAL_CAPITAL
            ax3.plot(
                bench_plot["Date"],
                bench_plot["Value"],
                linewidth=2,
                color="orange",
                alpha=0.7,
                label=f"{config.BENCHMARK}",
            )
    ax3.set_title("Strategy Performance", fontsize=14, fontweight="bold")
    ax3.set_ylabel("Portfolio Value ($)", fontsize=11)
    ax3.legend(loc="upper left")
    ax3.grid(True, alpha=0.3)
    ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
    ax4 = axes[3]
    returns = portfolio_df["Return"].dropna()
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    ax4.fill_between(
        portfolio_df["Date"].iloc[1:], drawdown.values * 100, 0, alpha=0.7, color="red"
    )
    ax4.set_title("Strategy Drawdown", fontsize=14, fontweight="bold")
    ax4.set_xlabel("Date", fontsize=11)
    ax4.set_ylabel("Drawdown (%)", fontsize=11)
    ax4.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\n" + "=" * 80)
    print("EXECUTION COMPLETE")
    print("=" * 80)
    return (results, regime_predictions, fold_list)


if __name__ == "__main__":
    results, regimes, folds = main()
